# REFIT Data Cleaning & Appliance Disaggregation Pipeline
**Take-Home Assignment for Flock Energy**

This reproducible notebook executes an end-to-end data science pipeline using the REFIT Electrical Load Measurements dataset (University of Strathclyde):
1. **Section 1: Basic Inspection & Cleaning of Raw Data (House 1)**: Timestamps parsing, ordering, de-duplication, irregular intervals, extreme spike removal, 1-minute resampling, and outage flagging.
2. **Section 2: Exploratory Analysis of Washing-Machine Use (5 Households)**: Data availability, quality, cycle detection rules, usage patterns by hour/day, and cross-household comparison.
3. **Section 3: Appliance-Disaggregation Model**: Sequence-to-point sliding-window feature engineering, leak-free chronological train/val/test split, baselines benchmarking, and model evaluation (MAE, RMSE, SAE, F1).
4. **Section 4: Practical Implications & Utility Recommendations**: Demand flexibility, limits, low-frequency smart meter challenges, and cross-geography transfer.


In [ ]:
import os
import sys
import tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.environ["MPLCONFIGDIR"] = os.path.join(tempfile.gettempdir(), "matplotlib_flock")
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

# Import modular pipeline functions
from src.clean_raw import inspect_raw_data, run_cleaning_pipeline, plot_cleaning_comparison, plot_interval_distribution
from src.eda_washing_machine import run_washing_machine_eda
from src.disaggregation_model import train_and_evaluate_disaggregation


## 1. Basic Inspection and Cleaning of Raw Data (House 1)

In this section, we inspect the uncleaned raw telemetry from House 1 (`RAW_House1_Part1.csv`), evaluate irregular sampling steps, identify duplicate timestamps and integer-overflow spikes, and execute a documented 1-minute resampling and outage flagging pipeline.


In [ ]:
# Inspect raw data for House 1
raw_summary, df_raw = inspect_raw_data('RAW_House1_Part1.csv')

# Plot sampling interval distribution
plot_interval_distribution(df_raw, output_dir='figures')
display(Image('figures/s1_house1_interval_distribution.png'))


In [ ]:
# Run cleaning pipeline and plot before/after comparison
df_raw_sorted, df_cleaned, is_outage = run_cleaning_pipeline(df_raw, max_interp_mins=3)
plot_cleaning_comparison(df_raw_sorted, df_cleaned, output_dir='figures')
display(Image('figures/s1_house1_cleaning_comparison.png'))


## 2. Exploratory Analysis of Washing-Machine Use

We evaluate 5 households with dedicated washing machine sub-metering channels:
- **House 1**: Beko WMC6140 (Channel 5)
- **House 2**: LG F1289TD (Channel 2)
- **House 3**: Unknown Model (Channel 6)
- **House 5**: AEG L99695HWD (Channel 3)
- **House 16**: Bosch WAB24262GB/01 (Channel 5)

We apply our physics-grounded cycle detection rule:
- Active threshold: $P \ge 20$ W
- Inactive pause tolerance: up to 3 minutes between active pulses (drum rest / soak pauses)
- Minimum duration: $\ge 15$ minutes
- Minimum peak power: $\ge 500$ W (confirming water heating / spin cycle)


In [ ]:
# Run EDA across the 5 households
summary_df, combined_cycles = run_washing_machine_eda(output_dir='figures')
display(summary_df)


In [ ]:
# Display Exploratory Plots
print("--- 24-Hour Aggregate vs Washing Machine Trace ---")
display(Image('figures/s2_24h_trace.png'))

print("--- Detected Cycle Profiles ---")
display(Image('figures/s2_wm_cycles.png'))

print("--- Diurnal (Hour of Day) & Day of Week Usage Patterns ---")
display(Image('figures/s2_hourly_dow_patterns.png'))

print("--- Cross-Household Comparison of Cycle Duration, Energy, & Frequency ---")
display(Image('figures/s2_cross_house_comparison.png'))


## 3. Appliance-Disaggregation Model

We build a sequence-to-point disaggregation model to estimate washing machine power from household aggregate power at 1-minute resolution on House 2:
- **Validation Design**: Strict chronological train (70%), validation (10%), and test (20%) split preventing temporal leakage.
- **Baselines**:
  1. *Baseline 1: Always-Off* ($\hat{y}_t = 0$)
  2. *Baseline 2: Hourly Profile* ($\hat{y}_t = \mu_{hour}$)
- **Disaggregation Model**: Histogram Gradient Boosting Regressor trained with windowed sequence features (point power, lags/leads $[-15, +15]$ min, multi-scale rolling statistics, step changes).


In [ ]:
# Load House 2 clean 1-minute data
clean_h2 = pd.read_csv('CLEAN_House2.csv', usecols=['Time', 'Unix', 'Aggregate', 'Appliance2', 'Issues'])
clean_h2_valid = clean_h2.loc[clean_h2['Issues'] == 0].copy()
clean_h2_valid['dt'] = pd.to_datetime(clean_h2_valid['Time'])
df_h2_1m = clean_h2_valid.set_index('dt')[['Aggregate', 'Appliance2']].resample('1min').mean().interpolate(method='time', limit=3)

# Train and evaluate disaggregation model
results_df, model, eval_df = train_and_evaluate_disaggregation(
    df_h2_1m,
    target_col='Appliance2',
    house_name='House 2',
    output_dir='figures'
)

display(results_df)


In [ ]:
# Display Actual vs Predicted Disaggregation on Holdout Test Set
print("--- Actual vs Predicted Power (Evaluation Set) ---")
display(Image('figures/s3_actual_vs_predicted.png'))

print("--- Test Set Prediction Error Distribution ---")
display(Image('figures/s3_error_distribution.png'))
